# บทเรียนที่ 09 - รูปแบบการออกแบบเมตาคอกนิชัน


## Setup

สมุดบันทึกนี้สาธิตรูปแบบการออกแบบ Metacognition โดยใช้ Microsoft Agent Framework

**ข้อกำหนดเบื้องต้น:**
- กำหนดค่าการปรับใช้ Azure OpenAI ผ่านตัวแปรสภาพแวดล้อม
- ลงชื่อเข้าใช้ Azure CLI (`az login`) แล้ว


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Metacognition คืออะไร?

Metacognition คือ **การคิดเกี่ยวกับการคิด** ในบริบทของปัญญาประดิษฐ์ (AI) หมายถึงการสร้างตัวแทนที่สามารถ:

- **สะท้อนตนเอง** เกี่ยวกับผลลัพธ์และกระบวนการเหตุผลของตนเอง
- **ตรวจจับข้อผิดพลาด** และฟื้นตัวอย่างนุ่มนวลแทนที่จะล้มเหลวโดยเงียบๆ
- **ประเมิน** ว่าคำตอบของตนสมบูรณ์และเป็นประโยชน์หรือไม่
- **ปรับตัว** ยุทธศาสตร์เมื่อวิธีการเริ่มแรกไม่สำเร็จ (เช่น การเปลี่ยนไปใช้ระบบสำรอง)

ตัวแทน metacognitive ไม่ได้แค่ตอบคำถาม — แต่ควบคุมดูแลประสิทธิภาพของตนเองและปรับเปลี่ยนได้ทันที


## เครื่องมือหลักและเครื่องมือสำรอง

รูปแบบเมตาค็อกนิชันทั่วไปคือ **กลยุทธ์สำรอง** ตัวแทนจะลองใช้เครื่องมือหลักก่อน; หากล้มเหลว (เช่น ข้อผิดพลาด 404) ตัวแทนจะรับรู้ความล้มเหลวและเปลี่ยนไปใช้เครื่องมือสำรองอย่างโปร่งใส

สิ่งนี้สะท้อนถึงระบบในโลกความจริงที่บริการหลักอาจใช้งานไม่ได้ และตัวแทนต้องวินิจฉัยปัญหาด้วยตนเองก่อนเลือกเส้นทางทางเลือก

ด้านล่างนี้เราได้กำหนดสองเครื่องมือค้นหาเที่ยวบิน:
- **หลัก** — ครอบคลุมปารีส โตเกียว และบาร์เซโลนา
- **สำรอง** — ครอบคลุมเบอร์ลิน ซิดนีย์ และนิวยอร์กซิตี้


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ตัวแทนสะท้อนตนเองพร้อมการกู้คืนข้อผิดพลาด

ตัวแทนด้านล่างได้รับคำสั่งให้ลองใช้ระบบการบินหลักก่อน, ตรวจจับความล้มเหลว, และถอยกลับไปใช้ระบบสำรองอย่างโปร่งใส หลังจากแต่ละคำตอบ จะทำการประเมินตนเองอย่างสั้น ๆ ว่าตอบคำถามของผู้ใช้ครบถ้วนหรือไม่


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## รูปแบบการประเมินตนเอง

อีกแง่มุมหนึ่งของเมตาคอกนิชันคือ **การประเมินตนเอง**: ตัวแทนแยกต่างหาก (หรือเดียวกันในรอบที่สอง) จะตรวจสอบคำตอบเพื่อความครบถ้วน ถูกต้อง และเป็นประโยชน์

ด้านล่างนี้เราจะสร้างตัวแทน `ResponseEvaluator` ที่ให้คะแนนคำตอบของตัวแทนท่องเที่ยวในสามมิติ


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีการสร้าง **ตัวแทนเมตาค็อกนิทีฟ** โดยใช้ Microsoft Agent Framework:

- **การสะท้อนตนเอง**: ตัวแทนที่ติดตามการให้เหตุผลของตนเองและสื่อสารอย่างโปร่งใสเกี่ยวกับสิ่งที่เกิดขึ้น
- **การกู้คืนจากข้อผิดพลาดด้วยฟอล์แบ็ค**: รูปแบบเครื่องมือหลัก + สำรองที่ตัวแทนตรวจจับความล้มเหลว (เช่น ข้อผิดพลาด 404) และพยายามใช้แหล่งข้อมูลทางเลือกโดยอัตโนมัติ
- **การประเมินตนเอง**: ตัวแทนผู้ประเมินแยกต่างหากที่ให้คะแนนคำตอบในเรื่องความสมบูรณ์ ความถูกต้อง และความช่วยเหลือ

รูปแบบเหล่านี้ทำให้ตัวแทนมีความทนทาน โปร่งใส และน่าเชื่อถือ — คุณสมบัติที่สำคัญสำหรับการนำไปใช้งานจริง


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
